# v3 rebuild — two-tower fusion, corrected

The first v3 ablation run produced a **collapsed two-tower network** (macro-F1 0.074,
below the 0.2 chance floor). Two compounding bugs were found in the original training
loop (both now fixed in `src/rebuild/ablations.py`, `loso_twotower`):

1. **Missing values were never imputed.** The cleaned matrix still has ~328 missing
   cells (0.05%). PyTorch's `mean`/`std` are *not* NaN-aware, so a single NaN in a
   column turned that column's fold statistics into NaN, which poisoned the
   standardized input of every window in the fold -> all logits NaN ->
   `argmax` over NaN degenerates to class 0. That alone explains macro-F1 ~ 0.07
   ("always predict class 0"). Fix: per-fold training-median imputation *before*
   standardization (train-only statistic, no leakage).
2. **Early-stopping score was noisy/unstratified.** The validation slice was a random
   15% of an imbalanced train set and the early-stop macro-F1 was computed over
   whichever classes appeared there. Fix: *stratified* validation split + fixed-label
   macro-F1 (`labels=np.arange(n_class)`) for early stopping + class-weighted loss +
   per-fold RNG seeding.

This notebook runs the corrected training over the same 52 held-out operators, then
rebuilds `modality_ablation.csv` and `fusion_benefit.csv` from the already-computed
cached folds + this new two-tower result. No other model is re-run.


In [1]:
import sys, time, copy, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import f1_score

import torch

def _root(start=Path.cwd()):
    for d in [start, *start.parents]:
        if (d / "src" / "rebuild" / "models_registry.py").exists():
            return d
    raise SystemExit("run notebook from the module root or src/")
REPO = _root()
sys.path.insert(0, str(REPO / "src" / "rebuild"))

from ablations import load, loso_twotower, ABLATION_MODELS
OUT = REPO / "output" / "research_outputs" / "fusion_training" / "v3_rebuild"
ABL = OUT / "ablation"
print("repo:", REPO)
print("out :", OUT)

repo: /home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion
out : /home/g0amer/Desktop/thesis/01_Task_Recognition_Multimodal_Fusion/output/research_outputs/fusion_training/v3_rebuild


In [2]:
df, X, y, subjects, feats = load()
n_class = int(y.max()) + 1
print("rows", len(df), "| features", X.shape[1], "| subjects",
      len(np.unique(subjects)), "| classes", n_class)

cached = sorted(p.name for p in ABL.glob("oof_*.npz") if "two_tower" not in p.name)
print(f"{len(cached)} cached modality runs available:")
for c in cached:
    print("   ", c)

rows 5640 | features 128 | subjects 52 | classes 5
12 cached modality runs available:
    oof_eeg_hist_gradient_boosting.npz
    oof_eeg_mlp.npz
    oof_eeg_random_forest.npz
    oof_eeg_xgboost.npz
    oof_fused_hist_gradient_boosting.npz
    oof_fused_mlp.npz
    oof_fused_random_forest.npz
    oof_fused_xgboost.npz
    oof_physio_hist_gradient_boosting.npz
    oof_physio_mlp.npz
    oof_physio_random_forest.npz
    oof_physio_xgboost.npz


### Run corrected two-tower (52 folds, ~4–8 min)

In [3]:
t0 = time.time()
mf_tt, yt_tt, yp_tt = loso_twotower(X, y, subjects, feats)
el = time.time() - t0
print(f"two-tower macro-F1 {mf_tt.mean():.4f} +/- {mf_tt.std(ddof=1):.4f}  "
      f"({el/60:.1f} min, 52 folds)")
print(f"pooled macro-F1 {f1_score(yt_tt, yp_tt, labels=np.arange(n_class), average='macro', zero_division=0):.4f}")
assert mf_tt.mean() > 0.20, "still collapsed — do not use"
np.savez_compressed(ABL / "oof_fused_two_tower.npz", y_true=yt_tt, y_pred=yp_tt, folds=mf_tt)
print("saved oof_fused_two_tower.npz")

two-tower macro-F1 0.4058 +/- 0.1575  (2.3 min, 52 folds)
pooled macro-F1 0.4512
saved oof_fused_two_tower.npz


### Rebuild `modality_ablation.csv` from cached folds + new two-tower

In [4]:
rows = []
for subset in ["eeg", "physio", "fused"]:
    for mn in ABLATION_MODELS:
        z = np.load(ABL / f"oof_{subset}_{mn}.npz")
        folds, yt, yp = z["folds"], z["y_true"], z["y_pred"]
        rows.append({"modality": subset, "model": mn,
                     "macro_f1_mean": float(folds.mean()),
                     "macro_f1_std": float(folds.std(ddof=1)),
                     "pooled_macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0))})
rows.append({"modality": "fused", "model": "two_tower",
             "macro_f1_mean": float(mf_tt.mean()),
             "macro_f1_std": float(mf_tt.std(ddof=1)),
             "pooled_macro_f1": float(f1_score(yt_tt, yp_tt, average="macro", zero_division=0))})
res = pd.DataFrame(rows)
res.to_csv(ABL / "modality_ablation.csv", index=False)
print(res.to_string(index=False))

modality                  model  macro_f1_mean  macro_f1_std  pooled_macro_f1
     eeg                xgboost       0.332968      0.109870         0.376968
     eeg hist_gradient_boosting       0.331648      0.117287         0.361875
     eeg          random_forest       0.282987      0.108103         0.331700
     eeg                    mlp       0.251708      0.112299         0.305458
  physio                xgboost       0.390449      0.152151         0.432641
  physio hist_gradient_boosting       0.401292      0.160795         0.430693
  physio          random_forest       0.382687      0.146547         0.440780
  physio                    mlp       0.336998      0.139927         0.379224
   fused                xgboost       0.462019      0.147869         0.503932
   fused hist_gradient_boosting       0.451087      0.146409         0.486600
   fused          random_forest       0.429959      0.132476         0.486995
   fused                    mlp       0.373561      0.159960    

### Rebuild `fusion_benefit.csv` (fused vs best single, paired Wilcoxon)

In [5]:
benefit = []
for mn in ABLATION_MODELS:
    f_fus = np.load(ABL / f"oof_fused_{mn}.npz")["folds"]
    best_sub, best_f = None, -np.inf
    for sub in ["eeg", "physio"]:
        f = np.load(ABL / f"oof_{sub}_{mn}.npz")["folds"]
        if f.mean() > best_f:
            best_f, best_sub = f.mean(), sub
    f_best = np.load(ABL / f"oof_{best_sub}_{mn}.npz")["folds"]
    d = f_fus - f_best
    try:
        w, p = stats.wilcoxon(d, zero_method="wilcox")
    except ValueError:
        w, p = float("nan"), float("nan")
    benefit.append({"model": mn, "best_single": best_sub,
                    "fused_mf": float(f_fus.mean()), "single_mf": float(f_best.mean()),
                    "delta": float(d.mean()), "wilcoxon_p": float(p)})
f_tt = np.load(ABL / "oof_fused_two_tower.npz")["folds"]
f_xgb = np.load(ABL / "oof_fused_xgboost.npz")["folds"]
d = f_tt - f_xgb
try:
    p_tt = stats.wilcoxon(d, zero_method="wilcox").pvalue
except ValueError:
    p_tt = float("nan")
benefit.append({"model": "two_tower_vs_xgboost_fused", "best_single": "xgboost_fused",
                "fused_mf": float(f_tt.mean()), "single_mf": float(f_xgb.mean()),
                "delta": float(d.mean()), "wilcoxon_p": float(p_tt)})
fb = pd.DataFrame(benefit)
fb.to_csv(ABL / "fusion_benefit.csv", index=False)
print(fb.to_string(index=False))

                     model   best_single  fused_mf  single_mf     delta  wilcoxon_p
                   xgboost        physio  0.462019   0.390449  0.071570    0.000113
    hist_gradient_boosting        physio  0.451087   0.401292  0.049795    0.001786
             random_forest        physio  0.429959   0.382687  0.047272    0.015035
                       mlp        physio  0.373561   0.336998  0.036564    0.105011
two_tower_vs_xgboost_fused xgboost_fused  0.405794   0.462019 -0.056225    0.001010


### Verdict

In [6]:
tt = float(fb.loc[fb.model == "two_tower_vs_xgboost_fused", "delta"].iloc[0])
xt = float(fb.loc[fb.model == "xgboost", "fused_mf"].iloc[0])
print(f"Two-Tower now {fb.loc[fb.model=='two_tower_vs_xgboost_fused','fused_mf'].iloc[0]:.3f} vs "
      f"fused XGBoost {xt:.3f}  ->  delta {tt:+.3f}")
print("Wrote modality_ablation.csv and fusion_benefit.csv with corrected two-tower.")

Two-Tower now 0.406 vs fused XGBoost 0.462  ->  delta -0.056
Wrote modality_ablation.csv and fusion_benefit.csv with corrected two-tower.
